# Synthetic Floor - GPU Blender Renderer (Colab)

Renders progressive construction stages with Blender Cycles on a Colab GPU,
built for **long, interruptible free-T4 sessions**.

### What makes this safe for multi-hour renders
- **Frame-level resume:** frames are rendered one at a time and written immediately.
  If Colab disconnects at frame 137/240, re-running continues from frame 137 - it does
  **not** restart the stage. (Canonical Blender pattern: skip frames already on disk.)
- **Continuous Drive sync:** every frame, log, IFC, mesh, .blend and checkpoint is
  mirrored to Google Drive within ~60 s (background daemon) and at each stage boundary.
- **Live progress inside a stage:** `render_progress.json` is updated after every frame
  (done/total, %, ETA) and printed by the runner.
- **Atomic writes** (temp + os.replace) so a crash never corrupts saved state.

### Choose an example and where to start
| `CONFIG` | Building | Final floor |
|---|---|---|
| `config/scene.yaml` | original single room | raw slab |
| `config/scene_office.yaml` | office room | tile |
| `config/scene_loft.yaml` | loft studio | wood |
| `config/scene_warehouse.yaml` | warehouse bay | epoxy |

Set `CONFIG` (which example) and optionally `START_STAGE` (1-7) below. Construction
logic: doors/windows are installed only at stage 6.

Set the runtime to **GPU**: *Runtime -> Change runtime type -> T4 (or A100)*.


## 1. Verify GPU, mount Drive, clone the repo, choose example + start stage


In [ ]:
import os, sys, subprocess

RUN_NAME    = 'demo'                       # same name => RESUME a previous run
CONFIG      = 'config/scene_office.yaml'   # which example to render
START_STAGE = 1                            # 1..7: render this stage -> 7 (resume-friendly)

try:
    print('GPU:', subprocess.check_output(['nvidia-smi','-L']).decode().strip())
except Exception:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4/A100.')

DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = f'/content/drive/MyDrive/MyCon_Colab/synthetic_floor_7stage/{RUN_NAME}'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive output folder:', DRIVE_ROOT)
except Exception as exc:
    print('Drive not available (not on Colab?):', exc)

if not os.path.isdir('/content/MyCon'):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/sayyedalimrj/MyCon.git','/content/MyCon'])
else:
    subprocess.call(['git','-C','/content/MyCon','pull','--ff-only'])
os.chdir('/content/MyCon/repository')
EXAMPLE = 'examples/synthetic_floor_7stage'
sys.path.insert(0, EXAMPLE + '/src')
print('cwd:', os.getcwd(), '| config:', CONFIG, '| start stage:', START_STAGE)


## 2. Install Blender 4.2 LTS


In [ ]:
%%bash
set -e
BLENDER_VERSION="4.2.3"
BLENDER_TAR="blender-${BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL="https://download.blender.org/release/Blender4.2/${BLENDER_TAR}"
if [ ! -f "/content/blender/blender" ]; then
  wget -q "${BLENDER_URL}" -O /tmp/blender.tar.xz
  mkdir -p /content/blender
  tar -xf /tmp/blender.tar.xz -C /content/blender --strip-components=1
  rm /tmp/blender.tar.xz
fi
apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 libegl1 libxfixes3 > /dev/null 2>&1 || true
/content/blender/blender --version 2>&1 | head -2


## 3. Install Python dependencies


In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy


## 4. Resolve the example's output folder


In [ ]:
from synthetic_floor.scene_spec import load_scene_spec
from pathlib import Path
SPEC = load_scene_spec(Path(EXAMPLE) / CONFIG)
OUTPUT = SPEC.output.root
RENDER_DIR = OUTPUT / 'blender_renders'
print('project:', SPEC.project_name, '| output:', OUTPUT)
print('stages :', [f'{s.id}:{s.name}' for s in SPEC.stages])


## 5. Render - full pipeline, resumable, starting at START_STAGE

`--start-stage` renders `START_STAGE..7`; `--resume` skips finished stages and
continues partly-rendered ones **frame-by-frame**; `--save-blend` writes a
downloadable .blend per stage; outputs sync to Drive continuously.

**If you get disconnected, just re-run this exact cell** - it picks up where it stopped.


In [ ]:
!PYTHONPATH={EXAMPLE}/src \
    python3 {EXAMPLE}/scripts/run_blender_gpu.py \
        --config {EXAMPLE}/{CONFIG} \
        --blender /content/blender/blender \
        --preset balanced --frames 120 --device OPTIX \
        --start-stage {START_STAGE} \
        --mount-drive --drive-root "{DRIVE_ROOT}" \
        --save-blend --resume --strict-render


## 6. Live intra-stage progress (frame-by-frame)

While stage N renders, this shows how many frames are done. Re-run it any time; it
also survives reconnects because `render_progress.json` is on Drive.


In [ ]:
import json
for stage_dir in sorted(RENDER_DIR.glob('stage_*')):
    pf = stage_dir / 'render_progress.json'
    if pf.exists():
        d = json.loads(pf.read_text())
        print(f"{stage_dir.name}: {d.get('frames_done')}/{d.get('frames_total')} "
              f"({d.get('percent')}%) status={d.get('status')} eta={d.get('eta_sec',0)/60:.1f}min")
    n = len(list((stage_dir / 'rgb').glob('frame_*.png'))) if (stage_dir/'rgb').exists() else 0
    print(f'   frames on disk: {n}')


## 7. Preview a frame + blank/over-exposure check


In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display
stage_dirs = sorted(RENDER_DIR.glob('stage_*'))
rgb_dir = (stage_dirs[-1] / 'rgb') if stage_dirs else RENDER_DIR
pngs = sorted(rgb_dir.glob('frame_*.png'))
print(f'{len(pngs)} frame(s) in {rgb_dir}')
if pngs:
    mid = pngs[len(pngs)//2]
    arr = np.asarray(Image.open(mid).convert('RGB'), dtype=np.float32)
    lum = arr.mean(axis=2); white = float((lum > 250).mean())
    print(f'{mid.name}: mean={lum.mean():.1f} std={lum.std():.2f} white_frac={white:.3f}')
    print('Looks OK' if (white < 0.97 and lum.std() > 3.0) else 'LOOKS BLANK - check render log')
    display(Image.open(mid))


## 8. Resume status, IFC (for the main pipeline) + camera path


In [ ]:
import json
rs = OUTPUT / 'manifests' / 'run_state_blender_gpu.json'
if rs.exists():
    body = json.loads(rs.read_text())
    print('run_id:', body.get('run_id'), '| in_progress:', body.get('in_progress'))
    for sid, st in sorted(body.get('stages', {}).items()):
        dm = st.get('done_marker') or {}
        print(f"  stage {sid}: complete={st['complete']} frames={dm.get('rgb_frame_count')} blend={bool(dm.get('blend_zip'))}")
ifc = OUTPUT / 'bim' / 'stage_07.ifc'
if ifc.exists():
    try:
        import ifcopenshell
        m = ifcopenshell.open(str(ifc))
        print('IFC4 entities:', {c: len(m.by_type(c)) for c in ('IfcFooting','IfcColumn','IfcBeam','IfcWall','IfcWindow','IfcDoor')})
    except Exception as exc:
        print('IFC present, ifcopenshell not loaded:', exc)


## 9. Download videos + .blend projects


In [ ]:
from IPython.display import FileLink, display
print('Videos:')
for mp4 in sorted((OUTPUT / 'video').glob('*.mp4')):
    print(f'  {mp4.name} ({mp4.stat().st_size/1024/1024:.1f} MB)'); display(FileLink(str(mp4)))
print('\nBlender projects (.blend zips):')
for z in sorted((OUTPUT / 'blend').glob('*.zip')):
    print(f'  {z.name} ({z.stat().st_size/1024/1024:.1f} MB)'); display(FileLink(str(z)))


## 10. How resume, stage selection & syncing work

**Checkpointing**
- Per-stage `.done` markers + a portable `manifests/run_state_blender_gpu.json`.
- Inside a stage, `render_progress.json` + the individual frame files are the
  fine-grained checkpoint: completed frames are never re-rendered.

**Drive syncing**
- Outputs mirror to `MyDrive/MyCon_Colab/synthetic_floor_7stage/<RUN_NAME>/output/`
  every ~60 s and at each stage boundary; all writes are atomic.

**Resume after interruption**
- Re-run cell 5 with the same `RUN_NAME` + `CONFIG`. Finished stages are skipped;
  a partly-rendered stage continues from its first missing frame.

**Choose which stage to start from**
- Set `START_STAGE` (1-7). `--start-stage 5` renders stages 5..7 only. It does not
  force a restart from stage 1.

**Choose which example**
- Set `CONFIG` to one of the four scenes. Each has its own output folder, so you can
  render several without overwriting each other.

**Continue on another device / Drive account**
- Copy that run folder to the new Drive, set the same `RUN_NAME` + `CONFIG`, and run.
